# 🛒 search-expert × ChromaDB — Hybrid Ecommerce Search

This notebook demonstrates a **hybrid search pipeline** for ecommerce:

<a target="_blank" href="https://colab.research.google.com/github/sarthakrastogi/search-expert/blob/main/examples/ecommerce/ecommerce_hybrid_search.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

```
Natural language query
        │
        ▼
search-expert-0.8b-json, a fine-tuned SLM
   → structured JSON with hard constraints  ──► metadata pre-filter
        │
        ▼
ChromaDB vector search (all-MiniLM-L6-v2)
   → semantic similarity over filtered set
        │
        ▼
Ranked results
```

**Why not just text-to-SQL?**  
Product descriptions like *'noise cancelling feel'* or *'good for travel'* live in unstructured text — SQL can't rank those. The structured filter handles hard constraints (price, brand, color); the vector search handles soft semantic intent. Together they're more powerful than either alone.

---


| | Text-to-SQL | Pure vector search | Hybrid search (this pipeline) |
|---|---|---|---|
| Hard constraints (price, brand, color) | ✅ | ❌ | ✅ |
| Semantic intent ("good for travel") | ❌ | ✅ | ✅ |
| Ranked results by relevance | ❌ | ✅ | ✅ |
| Works on unstructured descriptions | ❌ | ✅ | ✅ |
| Respects exclusions ("not black") | ✅ | ❌ | ✅ |
| Price is a hard cutoff, not a soft signal | ✅ | ❌ | ✅ |

Text-to-SQL is a **lookup tool** — it returns rows that match, but can't rank by relevance or understand semantics.  
Pure vector search is a **semantic tool** — it understands meaning, but treats "$200" as a soft hint, not a hard rule. A $350 product can rank above a $180 one if its description is more similar to the query.  
This pipeline is a **retrieval tool** — structured filters enforce the hard constraints first, then vector search ranks the surviving candidates by semantic relevance.

In production, we generally use a version of this pattern:  
**structured pre-filtering → ANN (approximate nearest neighbour) vector search → learning-to-rank re-ranker**.

search-expert makes step 1 trivial with a tiny, fast, locally-runnable model.


> IMPORTANT: For best speed, enable a GPU instance on Colab via Edit > Notebook Settings > t4 GPU


## 1 · Install dependencies

In [ ]:
%%capture
!pip install -q search-expert chromadb

## 2 · Download example files from GitHub

In [ ]:
BASE = "https://raw.githubusercontent.com/sarthakrastogi/search-expert/main/examples/ecommerce"

!wget -q {BASE}/products.py  -O products.py
!wget -q {BASE}/build_db.py  -O build_db.py
!wget -q {BASE}/search.py    -O search.py

print("✅  Files ready")

## 3 · Browse the product catalog

16 products across headphones, earbuds, laptops, and running shoes — each with a free-text description *and* structured metadata.

In [ ]:
import pandas as pd
from products import PRODUCTS

rows = []
for p in PRODUCTS:
    row = {"id": p["id"], "description": p["description"][:70] + "..."}
    row.update(p["metadata"])
    rows.append(row)

df = pd.DataFrame(rows)
df.head(5)

## 4 · Build the ChromaDB vector index

Each product description is embedded with **all-MiniLM-L6-v2** (runs locally, no API key needed).  
Structured metadata is stored alongside for hard filtering.

In [ ]:
import build_db

build_db.build()

## 5 · How the pipeline works

Before running searches, let's inspect each stage separately.

### 5a · Stage 1 — search-expert parses the query into structured JSON

In [ ]:
import json

from search_expert import SearchExpert

expert = SearchExpert()

queries = [
    "noise cancelling headphones any colour but black, under $200",
    "Sony wireless earbuds with hi-res audio under $300",
    "budget laptop under $500",
    "Nike running shoes under $150 in blue",
]

for q in queries:
    result = expert.parse(q)
    print(f"Query : {q}")
    print(f"Parsed: {json.dumps(result.fields, indent=2)}")
    print()

### 5b · Stage 2 — structured JSON → ChromaDB metadata filter

In [ ]:
from search import build_chroma_where

sample_parsed = {
    "domain": "ecommerce",
    "product": "headphones",
    "feature": "noise cancelling",
    "color": ["ne:black"],
    "price": "lt:200",
}

where = build_chroma_where(sample_parsed)
print("ChromaDB where-clause:")
print(json.dumps(where, indent=2))

The filter handles **hard constraints** (price < 200, exclude black) as exact metadata operations — no semantic ambiguity.

### 5c · Stage 3 — vector search within the filtered set

The query is embedded and compared against product description embeddings **only within the filtered subset**.  
This is what surfaces semantically relevant results (e.g. "good for travel" matching "compact, foldable").

## 6 · Run hybrid searches end-to-end

In [ ]:
from search import hybrid_search


def show_results(query, n=4):
    results = hybrid_search(query, n_results=n)
    print(f"\n{'═'*72}")
    print(f"Top {n} results:")
    for i, r in enumerate(results, 1):
        print(
            f"  {i}. [{r.score:.3f} sim]  {r.metadata.get('brand')} — {r.description[:80]}..."
        )
        print(
            f"       price=${r.metadata['price']:.2f}  rating={r.metadata['rating']}★  color={r.metadata['color']}"
        )
    print()

In [ ]:
# Hard constraint: price < $200, exclude black color
# Soft constraint: noise cancelling, headphones
show_results("noise cancelling headphones any colour but black, under $200")

In [ ]:
# Brand + feature combo
show_results("Sony wireless earbuds with hi-res audio under $300")

In [ ]:
# Pure price filter + category
show_results("budget laptop under $500")

In [ ]:
# Color + brand + price filter
show_results("Nike running shoes under $150 in blue")

In [ ]:
# Try your own query!
show_results("professional headphones for long meetings under $400")